# 08. Engagement 민감도 분석 + 긍정률 변화 분석

**입력**
- `data/review_individual.csv` — 개별 리뷰 (playtime, voted_up 포함)
- `data/discount_history.csv` — 할인 이벤트

**분석 3가지**
1. **Engagement 민감도 분석** — 플레이타임 필터 3기준(전체 / 2시간+ / 5시간+)별 Engagement 반응률·유지율
2. **긍정률 변화 (Sentiment Shift)** — 할인 전·중·후 구간별 voted_up 비율 변화
3. **장르별 플레이타임 분포** — 장르 분류의 데이터 기반 근거

**출력**: `figures/chart8~11.png`

In [2]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings('ignore')

def root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent]:
        if (c / 'data').exists() and (c / 'figures').exists():
            return c
    return cwd

PROJECT_ROOT = root()
DATA_DIR     = PROJECT_ROOT / 'data'
FIGURE_DIR   = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)

rv   = pd.read_csv(DATA_DIR / 'review_individual.csv')
disc = pd.read_csv(DATA_DIR / 'discount_history.csv')

disc['discount_start'] = pd.to_datetime(disc['discount_start'])
disc['discount_end']   = pd.to_datetime(disc['discount_end'])
rv['date'] = pd.to_datetime(rv['timestamp_created'], unit='s', utc=True).dt.tz_localize(None).dt.normalize()

print(f'리뷰: {len(rv):,}개 / 게임: {rv["app_id"].nunique()}개')
print(f'할인 이벤트: {len(disc)}개 / 게임: {disc["appid"].nunique()}개')

리뷰: 2,574,740개 / 게임: 55개
할인 이벤트: 808개 / 게임: 55개


In [3]:
# 폰트 설정
def kfont():
    for fp in [PROJECT_ROOT / 'fonts' / 'NanumGothic.ttc',
               PROJECT_ROOT / 'fonts' / 'NanumGothic.ttf']:
        if fp.exists():
            font_manager.fontManager.addfont(str(fp))
            return font_manager.FontProperties(fname=str(fp)).get_name()
    for fn in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']:
        if fn in {f.name for f in font_manager.fontManager.ttflist}:
            return fn
    return 'DejaVu Sans'

FONT_NAME = kfont()
plt.rcParams.update({'font.family': FONT_NAME, 'axes.unicode_minus': False, 'figure.dpi': 120})

GENRE_ORDER  = ['RPG', 'Adventure', 'Strategy/Simulation', 'Casual/Lightweight', 'Action']
GENRE_COLORS = {
    'RPG':                  '#4C72B0',
    'Adventure':            '#DD8452',
    'Strategy/Simulation':  '#55A868',
    'Casual/Lightweight':   '#C44E52',
    'Action':               '#8172B2',
}
DPI = 300

PRE_DAYS  = 30
POST_DAYS = 14
MIN_PRE_DAYS  = 14
MIN_POST_DAYS = 7

print(f'폰트: {FONT_NAME}')

# 게임별 리뷰 시계열 사전 (app_id → date-indexed series)
rv_by_app = {appid: grp.set_index('date')['review_id']
             for appid, grp in rv.groupby('app_id')}
print('게임별 리뷰 시계열 준비 완료')

폰트: Nanum Gothic
게임별 리뷰 시계열 준비 완료


## 분석 1 — Engagement 민감도 분석

플레이타임 필터 3가지 기준으로 동일한 Engagement 반응률·유지율을 계산하여,
필터에 따라 결론이 바뀌는지 확인한다.

| 기준 | 포함 대상 |
|------|----------|
| A: 전체 | 필터 없음 |
| B: 2시간+ | playtime_at_review_min ≥ 120 (Steam 환불 기준) |
| C: 10시간+ | playtime_at_review_min ≥ 600 (충분히 경험한 유저) |

In [4]:
def window_avg(series, start, end):
    mask = (series.index >= start) & (series.index < end)
    sub  = series[mask]
    if len(sub) == 0:
        return np.nan
    return sub.sum() / (end - start).days


def clip95(s):
    lo, hi = np.percentile(s.dropna(), [5, 95])
    return s.clip(lo, hi)


def compute_engagement(rv_filtered, disc):
    """개별 리뷰 → 일별 집계 → 이벤트별 Engagement 반응률·유지율 계산."""
    # 일별 집계
    daily = (rv_filtered.groupby(['app_id', 'date'])['review_id']
             .count().reset_index()
             .rename(columns={'review_id': 'cnt', 'app_id': 'appid'}))
    daily['date'] = pd.to_datetime(daily['date'])

    series_map = {appid: grp.set_index('date')['cnt']
                  for appid, grp in daily.groupby('appid')}

    rows = []
    for _, ev in disc.iterrows():
        appid = int(ev['appid'])
        if appid not in series_map:
            continue
        s     = series_map[appid]
        start = ev['discount_start']
        end   = ev['discount_end']
        pre_s = start - pd.Timedelta(days=PRE_DAYS)
        post_e = end + pd.Timedelta(days=POST_DAYS)

        if (start - s.index.min()).days < MIN_PRE_DAYS:
            continue
        if (s.index.max() - end).days < MIN_POST_DAYS:
            continue

        b = window_avg(s, pre_s, start)
        d = window_avg(s, start, end)
        a = window_avg(s, end, post_e)

        if pd.isna(b) or b == 0:
            continue

        rows.append({
            'appid':          appid,
            'name':           ev['name'],
            'genre_category': ev['genre_category'],
            'reaction_rate':  (d - b) / b if not pd.isna(d) else np.nan,
            'sustained_rate': (a - b) / b if not pd.isna(a) else np.nan,
        })

    result = pd.DataFrame(rows)
    if len(result):
        result['reaction_rate']  = clip95(result['reaction_rate'])
        result['sustained_rate'] = clip95(result['sustained_rate'])
    return result


print('함수 정의 완료')

함수 정의 완료


In [5]:
filters = {
    'A: 전체':    rv,
    'B: 2시간+':  rv[rv['playtime_at_review_min'] >= 120],
    'C: 10시간+': rv[rv['playtime_at_review_min'] >= 600],
}

results = {}
for label, rv_f in filters.items():
    df = compute_engagement(rv_f, disc)
    results[label] = df
    print(f'{label}: 유효 이벤트 {len(df)}개')

# 장르별 중앙값 집계
summary = {}
for label, df in results.items():
    summary[label] = (
        df.groupby('genre_category')[['reaction_rate', 'sustained_rate']]
        .median()
        .reindex(GENRE_ORDER)
        .round(3)
    )

print()
for label, s in summary.items():
    print(f'=== {label} ===')
    print(s.to_string())
    print()

A: 전체: 유효 이벤트 430개
B: 2시간+: 유효 이벤트 430개
C: 10시간+: 유효 이벤트 430개

=== A: 전체 ===
                     reaction_rate  sustained_rate
genre_category                                    
RPG                          0.241           0.054
Adventure                    0.275           0.091
Strategy/Simulation          0.200           0.039
Casual/Lightweight           0.251           0.105
Action                       0.379           0.109

=== B: 2시간+ ===
                     reaction_rate  sustained_rate
genre_category                                    
RPG                          0.218           0.056
Adventure                    0.248           0.116
Strategy/Simulation          0.182           0.039
Casual/Lightweight           0.228           0.101
Action                       0.346           0.102

=== C: 10시간+ ===
                     reaction_rate  sustained_rate
genre_category                                    
RPG                          0.136           0.067
Adventure            

In [6]:
filter_labels  = list(summary.keys())
filter_colors  = ['#4472C4', '#ED7D31', '#A9D18E']
x = np.arange(len(GENRE_ORDER))
w = 0.25

for metric, metric_label, fname in [
    ('reaction_rate',  'Engagement 반응률 중앙값',  'chart8_sensitivity_response.png'),
    ('sustained_rate', 'Engagement 유지율 중앙값', 'chart9_sensitivity_retention.png'),
]:
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (label, color) in enumerate(zip(filter_labels, filter_colors)):
        vals = summary[label][metric].values
        bars = ax.bar(x + (i - 1) * w, vals, w, label=label,
                      color=color, edgecolor='black', alpha=0.85)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                va  = 'bottom' if v >= 0 else 'top'
                off = 0.008 if v >= 0 else -0.008
                ax.text(bar.get_x() + bar.get_width() / 2, v + off,
                        f'{v:+.3f}', ha='center', va=va, fontsize=8)

    ax.axhline(0, color='gray', linestyle=':', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(GENRE_ORDER)
    ax.set_ylabel(metric_label)
    ax.set_title(f'플레이타임 필터별 {metric_label} (장르 × 기준)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / fname, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f'저장 완료 {fname}')

# 결과 표
print()
print('=== Engagement 반응률 민감도 분석 결과 ===')
tbl_r = pd.DataFrame({lbl: summary[lbl]['reaction_rate']  for lbl in filter_labels})
tbl_s = pd.DataFrame({lbl: summary[lbl]['sustained_rate'] for lbl in filter_labels})
print('[ 반응률 ]')
print(tbl_r.round(3).to_string())
print()
print('[ 유지율 ]')
print(tbl_s.round(3).to_string())

저장 완료 chart8_sensitivity_response.png
저장 완료 chart9_sensitivity_retention.png

=== Engagement 반응률 민감도 분석 결과 ===
[ 반응률 ]
                     A: 전체  B: 2시간+  C: 10시간+
genre_category                               
RPG                  0.241    0.218     0.136
Adventure            0.275    0.248     0.126
Strategy/Simulation  0.200    0.182     0.113
Casual/Lightweight   0.251    0.228     0.113
Action               0.379    0.346     0.204

[ 유지율 ]
                     A: 전체  B: 2시간+  C: 10시간+
genre_category                               
RPG                  0.054    0.056     0.067
Adventure            0.091    0.116     0.067
Strategy/Simulation  0.039    0.039     0.024
Casual/Lightweight   0.105    0.101     0.110
Action               0.109    0.102     0.058


## 분석 2 — 긍정률 변화 (Sentiment Shift)

할인 이벤트별로 직전 30일 / 할인 기간 / 종료 후 14일 구간의 `voted_up` 비율을 비교한다.

- 최소 표본 조건: 각 구간별 리뷰 **10개 이상**인 이벤트만 사용
- 표본 부족 이벤트는 별도 집계 후 장르 단위 보조 분석으로 제시

In [7]:
MIN_REVIEWS = 10

rv_by_appid = {appid: grp for appid, grp in rv.groupby('app_id')}

sent_rows = []
skipped   = 0

for _, ev in disc.iterrows():
    appid = int(ev['appid'])
    if appid not in rv_by_appid:
        skipped += 1
        continue

    g      = rv_by_appid[appid]
    start  = ev['discount_start']
    end    = ev['discount_end']
    pre_s  = start - pd.Timedelta(days=PRE_DAYS)
    post_e = end   + pd.Timedelta(days=POST_DAYS)

    before = g[(g['date'] >= pre_s)  & (g['date'] < start)]
    during = g[(g['date'] >= start)  & (g['date'] < end)]
    after  = g[(g['date'] >= end)    & (g['date'] < post_e)]

    if len(before) < MIN_REVIEWS or len(during) < MIN_REVIEWS:
        skipped += 1
        continue

    sent_rows.append({
        'appid':            appid,
        'name':             ev['name'],
        'genre_category':   ev['genre_category'],
        'before_positive':  before['voted_up'].mean(),
        'during_positive':  during['voted_up'].mean(),
        'after_positive':   after['voted_up'].mean() if len(after) >= MIN_REVIEWS else np.nan,
        'n_before':         len(before),
        'n_during':         len(during),
        'n_after':          len(after),
    })

sent_df = pd.DataFrame(sent_rows)
sent_df['sentiment_shift'] = sent_df['during_positive'] - sent_df['before_positive']

total_events = len(disc)
print(f'총 이벤트 {total_events}건 중 {len(sent_df)}건 분석 가능 (표본 부족 제외: {skipped}건)')
print()

# 장르별 집계
genre_sent = (
    sent_df.groupby('genre_category')
    .agg(
        직전30일=('before_positive', 'mean'),
        할인기간=('during_positive', 'mean'),
        종료후14일=('after_positive', 'mean'),
        변화량=('sentiment_shift', 'mean'),
        이벤트수=('appid', 'count'),
    )
    .reindex(GENRE_ORDER)
    .round(3)
)
print('=== 장르별 긍정률 변화 ===')
print(genre_sent.to_string())

총 이벤트 808건 중 449건 분석 가능 (표본 부족 제외: 359건)

=== 장르별 긍정률 변화 ===
                     직전30일   할인기간  종료후14일    변화량  이벤트수
genre_category                                        
RPG                  0.895  0.895   0.900 -0.000    65
Adventure            0.912  0.906   0.909 -0.006   125
Strategy/Simulation  0.912  0.918   0.924  0.006    83
Casual/Lightweight   0.902  0.901   0.899 -0.000   108
Action               0.863  0.844   0.862 -0.019    68


In [8]:
np.random.seed(42)

def bootstrap_ci_mean(values, n_boot=2000, ci=95):
    boot = [np.mean(np.random.choice(values, len(values), replace=True)) for _ in range(n_boot)]
    lo, hi = np.percentile(boot, [(100-ci)/2, 100-(100-ci)/2])
    return np.mean(values), lo, hi

# sentiment shift CI 계산
shift_ci = {}
for g in GENRE_ORDER:
    vals = sent_df[sent_df['genre_category']==g]['sentiment_shift'].dropna().values
    if len(vals) > 1:
        med, lo, hi = bootstrap_ci_mean(vals)
        shift_ci[g] = (med, lo, hi, len(vals))
    else:
        shift_ci[g] = (np.nan, np.nan, np.nan, len(vals))

# 2패널: 왼쪽 긍정률 전/중/후 / 오른쪽 변화량 + CI
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 왼쪽: 기존 bar chart
periods    = ['직전30일', '할인기간', '종료후14일']
period_clr = ['#5B9BD5', '#ED7D31', '#A9D18E']
x = np.arange(len(GENRE_ORDER))
w = 0.25
ax = axes[0]
for i, (period, color) in enumerate(zip(periods, period_clr)):
    vals = genre_sent[period].values
    bars = ax.bar(x + (i - 1) * w, vals, w, label=period,
                  color=color, edgecolor='black', alpha=0.85)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.004,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(GENRE_ORDER, rotation=8)
ax.set_ylabel('긍정률 (voted_up 비율)')
ax.set_title('장르별 할인 전·중·후 긍정률')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# 오른쪽: 변화량 (during - before) + 부트스트랩 95% CI
ax2 = axes[1]
for i, g in enumerate(GENRE_ORDER):
    med, lo, hi, n = shift_ci[g]
    color = GENRE_COLORS[g]
    ax2.bar(i, med, color=color, edgecolor='black', alpha=0.8)
    if not np.isnan(lo):
        ax2.errorbar(i, med, yerr=[[med - lo], [hi - med]],
                     fmt='none', color='black', capsize=6, linewidth=1.8)
    va  = 'bottom' if med >= 0 else 'top'
    off = 0.006 if med >= 0 else -0.006
    ax2.text(i, (hi if not np.isnan(hi) else med) + 0.006,
             f'{med:+.3f}\n(n={n})', ha='center', va='bottom', fontsize=8)

ax2.axhline(0, color='gray', linestyle=':', linewidth=1)
ax2.set_xticks(range(len(GENRE_ORDER)))
ax2.set_xticklabels(GENRE_ORDER, rotation=8)
ax2.set_ylabel('긍정률 변화량 (할인 중 - 직전 30일)')
ax2.set_title('장르별 긍정률 변화 (부트스트랩 95% CI)')
ax2.text(0.02, 0.02,
         '오차막대: 부트스트랩 95% CI (2,000회)\n0 위: 할인 중 긍정률 상승 / 0 아래: 하락',
         transform=ax2.transAxes, va='bottom', fontsize=8, color='gray')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart10_sentiment.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart10_sentiment.png')

print()
print('=== 긍정률 변화량 + 95% CI ===')
for g in GENRE_ORDER:
    med, lo, hi, n = shift_ci[g]
    ci_str = f'[{lo:+.4f}, {hi:+.4f}]' if not np.isnan(lo) else 'CI 없음'
    sig = '0 포함' if (not np.isnan(lo) and lo <= 0 <= hi) else ('양의 방향' if not np.isnan(lo) and lo > 0 else '음의 방향' if not np.isnan(hi) and hi < 0 else '')
    print(f'  {g:<25} 변화량={med:+.4f}  95% CI {ci_str}  ({sig})')

저장 완료 chart10_sentiment.png

=== 긍정률 변화량 + 95% CI ===
  RPG                       변화량=-0.0003  95% CI [-0.0092, +0.0110]  (0 포함)
  Adventure                 변화량=-0.0061  95% CI [-0.0110, -0.0011]  (음의 방향)
  Strategy/Simulation       변화량=+0.0060  95% CI [-0.0064, +0.0195]  (0 포함)
  Casual/Lightweight        변화량=-0.0005  95% CI [-0.0070, +0.0062]  (0 포함)
  Action                    변화량=-0.0187  95% CI [-0.0419, +0.0004]  (0 포함)


## 분석 2 확장 — event_sentiment.csv 생성

긍정/부정 리뷰 유형 필터 + 플레이타임 필터를 위한 이벤트별 데이터 사전 집계.

- 이벤트 기준: `analysis_df.csv` (263건, 유효 이벤트)
- 플레이타임 3종: 전체 / 2시간+ / 10시간+
- 출력: `data/event_sentiment.csv` (최대 789행)
- merge 키: `appid` + `discount_start` + `playtime_filter`

In [9]:
PLAYTIME_THRESHOLDS = {'all': 0, '2h': 120, '10h': 600}
SENTIMENT_THRESHOLD = 0.005  # +-0.5%p 이내 -> 'neutral'

analysis_ev = pd.read_csv(DATA_DIR / 'analysis_df.csv')
analysis_ev['discount_start'] = pd.to_datetime(analysis_ev['discount_start'])
analysis_ev['discount_end']   = pd.to_datetime(analysis_ev['discount_end'])

export_rows = []

for pt_label, min_pt in PLAYTIME_THRESHOLDS.items():
    rv_pt        = rv[rv['playtime_at_review_min'] >= min_pt]
    rv_by_app_pt = {appid: grp for appid, grp in rv_pt.groupby('app_id')}
    print(f'[{pt_label}] 리뷰 {len(rv_pt):,}건 필터 완료')

    for _, ev in analysis_ev.iterrows():
        appid = int(ev['appid'])
        if appid not in rv_by_app_pt:
            continue

        g     = rv_by_app_pt[appid]
        start = ev['discount_start']
        end   = ev['discount_end']
        pre_s = start - pd.Timedelta(days=PRE_DAYS)

        before = g[(g['date'] >= pre_s) & (g['date'] < start)]
        during = g[(g['date'] >= start) & (g['date'] < end)]

        if len(before) < MIN_REVIEWS or len(during) < MIN_REVIEWS:
            continue

        before_days = PRE_DAYS
        during_days = max((end - start).days, 1)

        bp = int(before['voted_up'].sum())
        bn = int((~before['voted_up']).sum())
        dp = int(during['voted_up'].sum())
        dn = int((~during['voted_up']).sum())

        b_pos_avg = bp / before_days
        b_neg_avg = bn / before_days
        d_pos_avg = dp / during_days
        d_neg_avg = dn / during_days

        rr_pos = (d_pos_avg - b_pos_avg) / b_pos_avg if b_pos_avg > 0 else np.nan
        rr_neg = (d_neg_avg - b_neg_avg) / b_neg_avg if b_neg_avg > 0 else np.nan

        pr_before = bp / len(before) if len(before) >= MIN_REVIEWS else np.nan
        pr_during = dp / len(during) if len(during) >= MIN_REVIEWS else np.nan

        if pd.isna(pr_before) or pd.isna(pr_during):
            pr_delta = np.nan
            sg       = 'insufficient_data'
        else:
            pr_delta = pr_during - pr_before
            if pr_delta > SENTIMENT_THRESHOLD:
                sg = 'up'
            elif pr_delta < -SENTIMENT_THRESHOLD:
                sg = 'down'
            else:
                sg = 'neutral'

        export_rows.append({
            'appid':                  appid,
            'discount_start':         start.strftime('%Y-%m-%d'),
            'playtime_filter':        pt_label,
            'positive_count_before':  bp,
            'negative_count_before':  bn,
            'positive_count_during':  dp,
            'negative_count_during':  dn,
            'before_days':            before_days,
            'during_days':            during_days,
            'response_rate_positive': round(rr_pos, 6) if not pd.isna(rr_pos) else np.nan,
            'response_rate_negative': round(rr_neg, 6) if not pd.isna(rr_neg) else np.nan,
            'positive_rate_before':   round(pr_before, 6) if not pd.isna(pr_before) else np.nan,
            'positive_rate_during':   round(pr_during, 6) if not pd.isna(pr_during) else np.nan,
            'positive_rate_delta':    round(pr_delta, 6) if not pd.isna(pr_delta) else np.nan,
            'sentiment_group':        sg,
        })

event_sentiment = pd.DataFrame(export_rows)
event_sentiment.to_csv(DATA_DIR / 'event_sentiment.csv', index=False)
print()
print(f'저장 완료: data/event_sentiment.csv ({len(event_sentiment)}행)')
print()
print('=== 플레이타임 필터별 유효 이벤트 수 ===')
print(event_sentiment.groupby('playtime_filter').size().rename('이벤트 수').to_string())
print()
print('=== sentiment_group 분포 (all 필터) ===')
print(event_sentiment[event_sentiment['playtime_filter'] == 'all']['sentiment_group'].value_counts().to_string())

[all] 리뷰 2,574,740건 필터 완료
[2h] 리뷰 2,502,860건 필터 완료
[10h] 리뷰 1,979,552건 필터 완료

저장 완료: data/event_sentiment.csv (783행)

=== 플레이타임 필터별 유효 이벤트 수 ===
playtime_filter
10h    261
2h     261
all    261

=== sentiment_group 분포 (all 필터) ===
sentiment_group
down       123
up          75
neutral     63


## 분석 3 — 장르별 플레이타임 분포

장르 분류가 단순 태그가 아니라 **실제 소비 행동 차이**를 반영한다는 데이터 기반 근거.

- `playtime_at_review_min` 기준 (리뷰 작성 시점 플레이타임)
- playtime = 0인 리뷰 제외
- y축 로그 스케일

In [10]:
pt = rv[rv['playtime_at_review_min'] > 0].copy()
pt['playtime_hours'] = pt['playtime_at_review_min'] / 60

# 기초 통계
stats = (
    pt.groupby('genre')['playtime_hours']
    .agg(['median', 'mean',
          ('p25', lambda x: x.quantile(0.25)),
          ('p75', lambda x: x.quantile(0.75)),
          ('p95', lambda x: x.quantile(0.95))])
    .reindex(GENRE_ORDER)
    .round(1)
)
print('=== 장르별 플레이타임(시간) 분포 ===')
print(stats.to_string())
print()

# 박스플롯
data_by_genre = [
    pt[pt['genre'] == g]['playtime_hours'].values
    for g in GENRE_ORDER
]

fig, ax = plt.subplots(figsize=(11, 6))
bp = ax.boxplot(
    data_by_genre,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color='black', linewidth=2),
)
for patch, genre in zip(bp['boxes'], GENRE_ORDER):
    patch.set_facecolor(GENRE_COLORS[genre])
    patch.set_alpha(0.75)

ax.set_yscale('log')
ax.set_xticks(range(1, len(GENRE_ORDER) + 1))
ax.set_xticklabels(GENRE_ORDER)
ax.set_ylabel('리뷰 작성 시점 플레이타임 (시간, 로그 스케일)')
ax.set_title('장르별 플레이타임 분포 (playtime = 0 제외, 이상치 미표시)')
ax.grid(axis='y', alpha=0.3)

# 중앙값 레이블
for i, genre in enumerate(GENRE_ORDER, 1):
    med = stats.loc[genre, 'median']
    ax.text(i, med * 1.15, f'중앙값\n{med:.1f}h',
            ha='center', va='bottom', fontsize=8)

ax.text(0.98, 0.02, f'리뷰 {len(pt):,}개 (playtime>0)',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart11_playtime_by_genre.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart11_playtime_by_genre.png')

=== 장르별 플레이타임(시간) 분포 ===
                     median   mean   p25    p75    p95
genre                                                 
RPG                    50.9  121.2  16.4  124.8  439.2
Adventure              18.2   91.8   8.0   51.0  319.2
Strategy/Simulation    36.1  185.3  10.9  132.1  847.9
Casual/Lightweight     28.5   87.9  10.1   81.3  346.9
Action                 45.9  166.1  12.8  139.3  702.0



Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


저장 완료 chart11_playtime_by_genre.png
